# Understanding JAX Compilation and Tracing

JAX achieves high runtime performance by transforming Python functions into computational graphs using the XLA (Accelerated Linear Algebra) compiler. During compilation, JAX traces the operations performed by a function to build an optimized computation graph. This graph is then compiled once into efficient machine code.

After the initial compilation, subsequent executions reuse the compiled graph whenever the input shapes and types remain unchanged, significantly reducing overhead and enabling fast execution. This compilation model is a key factor behind JAX's performance and scalability on CPUs, GPUs, and TPUs.

In [1]:
import jax
import jax.numpy as jnp
from jax import jit, lax
import numpy as np
from fgpt.core.common import Logger

In [2]:
logger = Logger()
jax.config.update('jax_enable_x64', True) # This is needed to ensure that the float64 to be available 
                                          # since by default all the values goes to float32

In [3]:
# Jax.numpy mirrors that of the numpys as such all the operations that are done in numpy are also available in JAX
x = jnp.array([2,3,4],dtype=jnp.float32)
logger.info(f'Array: {x}, Type: {type(x)}')

[INFO] Array: [2. 3. 4.], Type: <class 'jaxlib._jax.ArrayImpl'>

In [4]:
# Jax arrays are immutable due to this operations such as +=, *= aren't directly possible
a = np.array([1,3,4,5],dtype=np.float32)
logger.info(f'Array a: {a[:]*10}')

b = jnp.array([1,3,4,5],dtype=jnp.float32)
logger.info(f'Array b: {b[:]*10}') # THis works but 

# b[0] = b[0] * 10 # THis will launch this error: TypeError: JAX arrays are immutable and do not support in-place item assignment.
b *= 10 # THIS is possible due to the fact that b is newly created variable that is created with values from the older variable b and the multiplication with 10
logger.info(f'Array b after multiplication: {b}')

[INFO] Array a: [10. 30. 40. 50.]

[INFO] Array b: [10. 30. 40. 50.]

[INFO] Array b after multiplication: [10. 30. 40. 50.]

In [5]:
# Jax doesn't throw the out of bound errors but simply clamps the index to the size of the array and any modification applied to the out of bound indexes are ignored. 
logger.info(f'Array b[10]: {b[10]}, Array b[-1]: {b[-1]}')

[INFO] Array b[10]: 50.0, Array b[-1]: 50.0

In [6]:

@jit
def modify(x):
    return x.at[0].set(10) # In place modification is not possible which means x[i] = ... is not possible so we do this 
    # https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html#jax.numpy.ndarray.at However, inside a jit() compiled function, expressions like x = x.at[idx].set(y) are guaranteed to be applied in-place.

x = jnp.array([1, 2, 3]) # Jax arrays are immutables due to the pure functionnal array case thus requires a return 
y = modify(x)
logger.info(f"Modificed y:{y}")

[INFO] Modificed y:[10  2  3]

In [7]:
@jit
def modify(x,a,b):
    for i in range(len(x)):
        x = x.at[i].set(a[i]/b) # JAX arrays are immutable, so `.at[i].set(...)` returns a new array
        # with the i-th element replaced by (a[i] / b)
        # Under @jit, JAX traces the entire function once to build a computation graph.
        # The loop is unrolled or converted into an equivalent JAX operation at compile time.
    return x

x = jnp.array([1., 2., 3.])
a = jnp.array([10.,20.,30.])
b = 5.0
y = modify(x,a,b)
logger.info(f"Modificed y:{y}")

[INFO] Modificed y:[2. 4. 6.]

In [8]:
@jit
def modify(x, a, b):
    # x.at[:] jax sees it as a flattened array and does the modification elemenetwise
    return x.at[:].set(a / b)

x = jnp.zeros((2, 3))        # shape (2, 3)
a = jnp.array([[1, 2, 3],    # shape (2, 3)
               [4, 5, 6]])
b = 2

modify(x, a, b)

Array([[0.5, 1. , 1.5],
       [2. , 2.5, 3. ]], dtype=float64)

In [9]:
@jit
def modify_conditional(x, a, b):
    # If an argument is directly this works without the use of @jit but when added it fails with a traceboolconversion error this is due to the fact that 
    # a[i] and b becomes a boolean traced array and the tracing forgets which path to backtrack since the thing we backtrack isn't the values but the shape and dtype 
    # for i in range(len(x)):
    #     if a[i] > b:
            
    #         x = x.at[i].set(a[i] / b)
    # Thus it requires the conversion of into a vectorized approach. This is because the input values are not known during the compilation
    # thus the jit does the compilation once and resuses it again to reevaluate for the all the rest of variables 
    x = jnp.where(a > b, a / b, x)
    return x

x = jnp.array([0, 0, 0],dtype=float)
a = jnp.array([1, 5, 3],dtype=float)
b = 3.

modify_conditional(x, a, b)

Array([0.        , 1.66666667, 0.        ], dtype=float64)

In [10]:
def f(x):
  if x < 3:
    return 3. * x ** 2
  else:
    return -4 * x

f = jit(f, static_argnames='x') # THe problem in here is that the x the static argument names changed a lot this would be bad since this means that for each new value the function is recompiled 
# It won't be necessary to apply jit in the case we do gradient 
logger.info(f(2.))

[INFO] 12.0

In [11]:
# One of the examples was that for static values to use numpy and jax.numpy for operations to be traced as such that numpy is done at compile time and 
# jax.numpy is done at both compile and run time 

@jit
def f(x):
  return x.reshape((np.prod(x.shape),))

x = jnp.ones((2, 3))
logger.info(f(x)) # We still have x as Jax array type

[INFO] [1. 1. 1. 1. 1. 1.]

In [12]:
@jit
def divide(x, y):
  logger.info(y)
  logger.info(y >= 1.)
  # return x / y if y >= 1. else 0.

#divide = jit(divide,static_argnums=1) # ANOTHER concept is that the y which is a regular python expression get's traced as int32 and the expression  y >= 1 becomes a bool thus 
# launches an error. The possibility is only to use static arguments but is deadly. Another reason being is that these are abstract tracers thus we only know the shape and dtype
# but there also exists the concrete tracers which also knws the values
logger.info(divide(3,2))

[INFO] Traced<~int64[]>with<DynamicJaxprTrace>

[INFO] Traced<~bool[]>with<DynamicJaxprTrace>

[INFO] None

In [13]:
# Function to apply if predicate is True
def true_fn(x):
    return x * 2

# Function to apply if predicate is False
def false_fn(x):
    return x + 10

# Wrapped conditional function
def compute(x):
    return lax.cond(
        x > 0,        # predicate (must be JAX-traceable)
        true_fn,      # called if True
        false_fn,     # called if False
        x             # input passed to both branches
    )

x = jnp.array(5)
logger.info(compute(x))  # 10

[INFO] 10

In [14]:
def step(carry, x):
    new_carry = carry + x
    return new_carry, new_carry  # return updated state and output

xs = jnp.array([1.0, 2.0, 3.0, 4.0])

# Initial carry = 0
init_carry = 0.0

final_carry, outputs = lax.scan(step, init_carry, xs)

logger.info(f"Final: {final_carry}")   # 10.0
logger.info(f"Outputs: {outputs}")     # [1, 3, 6, 10]

[INFO] Final: 10.0

[INFO] Outputs: [ 1.  3.  6. 10.]

In [15]:
def rnn_step(h, x):
    # simple recurrence: h_t = tanh(h + x)
    new_h = jnp.tanh(h + x)
    return new_h, new_h

xs = jnp.array([0.1, 0.2, 0.3, 0.4])

h0 = 0.0

final_h, hs = lax.scan(rnn_step, h0, xs)

logger.info(hs)

[INFO] [0.09966799 0.29100875 0.53062073 0.73088317]

In [16]:
def divide(x, y):
  return x / y if y >= 1. else 0.

logger.info(jax.grad(divide)(3.,2.)) # As we can see that there was no need for applying static argnums or argnames due to the fact that grad, jvp, and vjp uses concrete tracers
# BUt we also need to accelerate this we further which could be done using jit
%timeit jax.grad(divide)(3.,2.).block_until_ready()

[INFO] 0.5

810 μs ± 941 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [17]:
divide = jit(divide,static_argnums=1) # The problem here is that we are placing y as a static argunums
%timeit jax.grad(divide)(3.,2.).block_until_ready() # Performance increase of 814 / 736 = 110 %

764 μs ± 1.81 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


The issue is that the conditionals are quite difficult to modify and require special care, especially when they use the arguments being passed, as this can lead to conflicts.

In [18]:
seed = 0x123456789 # some integer seed.
key = jax.random.PRNGKey(seed)

In [19]:
def mask_tensor(x, mask):
  x = x.at[mask].set(-100.)
  return x

key, x_key, mask_key = jax.random.split(key, 3)
x = jax.random.normal(x_key, (4,4))
mask = jax.random.uniform(mask_key, (4,4)) < 0.5

%timeit mask_tensor(x,mask).block_until_ready()

890 μs ± 4.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [20]:
# BUT IF we add jit to this, it will cause an error due to the fact that mask shape is not known 
@jit
def mask_tensor(x, mask):
  x = ~mask * x - mask*100.
  return x

# All intermediate shapes will be known at compile time. To break it down, we multiply x by zero where mask is True, and by one where it is False. 
# We then add a new array that is zero where mask is False and -100 where mask is True. At this point we have two arrays with concrete shapes. 
# Adding them together yields the correct result, which is similarly concrete.

%timeit mask_tensor(x,mask).block_until_ready()


6.42 μs ± 35.5 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [21]:
@jit
def test(a,b):
    return lax.cond(b, lambda _: a, lambda _: 1, operand=None)
    
a = jnp.int64(2)
c = np.bool_(True) # This is traced at run time to check for the value. 

logger.info(test(a,c))

[INFO] 2

In [22]:
# THis wouldn't be case for our modules since all given arrays have their shapes and dtypes pre defined and known, the only thing that we do is that we iterate over them and modify them.
# COde that doesn't manipulate JAX arrays will not be traced and called only during the tracing itself. 

x = jnp.array([[1, 3, 4, 2],
               [5, 2, 6, 3],
               [8, 1, 3, 9]])

# with numpy this works as the following
logger.info(np.sum(x,axis=1)) # or with a for loop which would take even more time as if the original python code does have this the transformation process could be even longer.
# An attempt at transforming the  
# logger.info(jnp.sum(x,axis=1)) does the same as numpy 

[INFO] [10 16 21]

In [23]:
# Pytrees -> tree like structure built out of container like python objects, such as classes, this includes classes such as list, tuple, dict and any thing out of it is considered a single array,element;
# as such it becomes possible to use for loops but the condition aren't because this is due to the fact that lstep is given as the values of arguments but we can send the arrays initialized as arguments
# Might need to give a look when we attack per class  

dims = [784, 64, 10]
params = []
    
params.append({
    'W': jnp.zeros((dims[1], dims[0])),  # output_dim x input_dim
    'b': jnp.zeros((dims[1],)),          # output_dim
})

logger.info(f"\n{params}")

[INFO] 
[{'W': Array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float64), 'b': Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 
0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float64)}]

In [24]:
@jax.jit
def feed_forward(params, x):
    for p in params:
        x = jax.nn.tanh(p['W'] @ x + p['b'])
        # jax.nn.tanh(p['W'] @ x + p['b'])

    return x
    
key, x_key = jax.random.split(key)
# feed_forward(params, jax.random.normal(x_key, (dims[0],)))
x = jax.random.normal(x_key, (dims[0],))
%timeit feed_forward(params, x).block_until_ready()

24 μs ± 148 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [25]:
# jax.grad only works for scalar-ouput functions, it's also possible to send arguments(argnums) to differentiate a function with respect to an argument

f = lambda x: x**3 + 2*x**2 - 3*x + 1
dfdx = jit(jax.grad(f)) # It is possible to use jit with grad and the other way around as well: jax.grad(jit(f))
logger.info(dfdx(1.))

[INFO] 4.0

In [26]:
f = lambda x,y: x**3 + 2*x*y**2 - 3*x + y + 1

dfdx = jit(jax.grad(f,argnums=1)) # This will derive df/dy 
logger.info(dfdx(1.,1.))

[INFO] 5.0

In [27]:
# Define your function
def func(x, y):
    return x**3 + 2*x*y**2 - 3*x + y + 1

dfdx = jit(jax.grad(func))

logger.info(dfdx(1.0, 2.0))  # Works fine for scalars

# Now, if you we to handle arrays of x and y values,
# we can use vmap to vectorize across array elements and without the vmap the array use of function doesn't seem to work. 
dfdx_vectorized = jit(jax.vmap(dfdx))

x_vals = jnp.array([1.0, 2.0, 3.0])
y_vals = jnp.array([2.0, 3.0, 4.0])

res = dfdx_vectorized(x_vals, y_vals)
logger.info(res)

%timeit dfdx_vectorized(x_vals, y_vals).block_until_ready()

[INFO] 8.0

[INFO] [ 8. 27. 56.]

6.43 μs ± 6.43 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [28]:
# the vmap argument in_axes defines the axes upon which we need to compute, as such in_axes defines the indices upon which they need to be calculate
def compute(x, y):
    return x**2 + y

# Shape (N, M)
x = jnp.arange(6).reshape(2, 3)
y = jnp.ones_like(x)

# Two vmaps → one for each loop
batched_fn = jax.vmap(jax.vmap(compute, in_axes=(0, 0)), in_axes=(0, 0))
out = batched_fn(x, y)
logger.info(f"\n{out}")
%timeit batched_fn(x,y).block_until_ready() # Faster than having two for loops within the function

[INFO] 
[[ 1  2  5]
 [10 17 26]]

1.12 ms ± 988 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [29]:
@jit
def compute(x, y):
    return x**2 + y

batched_fn = jax.vmap(jax.vmap(compute, in_axes=(0, 0)), in_axes=(0, 0))
out = batched_fn(x, y)
%timeit batched_fn(x,y).block_until_ready() # ~50% faster runtime than with @jit, IN order to use @jit, we need to ensure that the elements are tracable 

646 μs ± 1.74 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [30]:
# jax.jvp also known as forward autodiff / tangent linear which requires the N evaluations for functions with N inputs, thus slower but faster in the case vjp, thus it's better to use 
# f(x+ev) ~= f(x) + df(x)*v where it takes a tuple (x, v) where the application of this tuple x on f(x) is the primal and the df(x)*v is called the tangent. 

# THe primals corresponds to hte input to the fucntions at which we evaluate the JVP function
# the tangents is the vector in the direction of the derivative that we are computing. 

# THe jvp computes the changes of the funciton in response to the response(primals) and the direction of the derivative. 
# jvp = J(u) * v where J(u) is the jacobian and v is the tangent direction 

def COST_FUNCTION(u):
    e = jnp.exp(1.0)
    j = (u - e)**2
    return j

u0 = 1.0          
v = 1.0       

j_val, jvp_val = jax.jvp(COST_FUNCTION,
                         (u0,),    
                         (v,))
logger.info(f"Function value: {j_val}") # This is just the cost_function itself evaluated for the scalar 
logger.info(f"JVP (derivative): {jvp_val}") # Gradient, since this is a scalar and this corresponds to the tangent linear model output

[INFO] Function value: 2.9524924420125607

[INFO] JVP (derivative): -3.436563656918091

In [31]:
u0 = jnp.array([1.0, 2.0, 3.0])
v  = jnp.ones_like(u0)
j_val, jvp_val = jax.jvp(COST_FUNCTION, (u0,), (v,))
logger.info("Using jax.jvp instead....")
logger.info(f"Function value: {j_val}")
logger.info(f"JVP: {jvp_val}") # DIrectional derivative since this is an array

# But in the case when the input is much larger like in neural networks, we can use the jax.grad but it's output should be that of a scalar, and in the case we use the jvp which means for each tangent values
# this will call the jvp at each tangent time. 
logger.info("Using jax.grad instead....")
grad_fn = jax.grad(COST_FUNCTION)
for i in u0:
    grad_val = grad_fn(i)
    logger.info(grad_val)

# Instead we can use jax.jacfwd which functions in the jax.grad but for vectors BUT gives out a vector format with evaluation done colum by column with forward AD but we will stick with jvp 
logger.info("Using jax.jacfwd instead....")
grad_fn = jax.jacfwd(COST_FUNCTION)
jvp_val = grad_fn(u0)
logger.info(f"JVP:\n {jvp_val}")

[INFO] Using jax.jvp instead....

[INFO] Function value: [2.95249244 0.51592879 0.07936513]

[INFO] JVP: [-3.43656366 -1.43656366  0.56343634]

[INFO] Using jax.grad instead....

[INFO] -3.436563656918091

[INFO] -1.436563656918091

[INFO] 0.5634363430819089

[INFO] Using jax.jacfwd instead....

[INFO] JVP:
 [[-3.43656366 -0.         -0.        ]
 [-0.         -1.43656366 -0.        ]
 [ 0.          0.          0.56343634]]

Thus it's possible to use jax.jvp and giving it a tangent value of the same shape of primals with values of ones, thus could be used to calculate the efficient parameters models during the tangent linear models calculation. 

In [32]:
# VJP which stands for the adjoint linear (Vector jacobian product) produces the same results as seen below for the function value and the vjp value where we can 
# see that that vjp val is the derivative function itself thus the same results as above 
rev_val, vjp_val = jax.vjp(COST_FUNCTION, 1.0)
logger.info(f"Function value: {rev_val}")
logger.info(f"VJP: {vjp_val((1.0))}")

[INFO] Function value: 2.9524924420125607

[INFO] VJP: (Array(-3.43656366, dtype=float64, weak_type=True),)

In [33]:
rev_val, vjp_val = jax.vjp(COST_FUNCTION, u0)
logger.info(f"Function value: {rev_val}")
logger.info(f"VJP: {vjp_val((u0))}")

%timeit vjp_val((u0))[0].block_until_ready()

[INFO] Function value: [2.95249244 0.51592879 0.07936513]

[INFO] VJP: (Array([-3.43656366, -2.87312731,  1.69030903], dtype=float64),)

218 μs ± 354 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


The main difference between `jvp` and `vjp` is that jvp computes directional derivatives directly during the forward pass for a given input and tangent vector. In contrast, vjp first performs the forward pass and records the necessary information before computing the backward pass.

Additionally, the second output returned by `vjp` is a function that represents the vector-Jacobian product. This function can then be called with a cotangent vector to evaluate the corresponding gradients or sensitivities.

In [34]:
jvp_val = jit(jax.jacrev(COST_FUNCTION)) # returns an array(matrix in this case) like answer containing the gradient evaluated row by row using the reverse mode AD. 
logger.info(f"JVP:\n{jvp_val(u0)}")

[INFO] JVP:
[[-3.43656366  0.          0.        ]
 [ 0.         -1.43656366  0.        ]
 [ 0.          0.          0.56343634]]

In [35]:
@jit
def COST_FUNCTION(u):
    e = jnp.exp(1.0)
    j = (u - e)**2
    return j

rev_val, vjp_val = jax.vjp(COST_FUNCTION, u0)
logger.info(f"Function value: {rev_val}")
logger.info(f"VJP: {vjp_val((u0))}")

%timeit vjp_val((u0))[0].block_until_ready() # THe thing about @jit or jit is that once the function is compiled, 
                                             # it doesn't need to recompile again due to the fact that the function itself is traced. 

[INFO] Function value: [2.95249244 0.51592879 0.07936513]

[INFO] VJP: (Array([-3.43656366, -2.87312731,  1.69030903], dtype=float64),)

129 μs ± 226 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [36]:
# Pytrees custom 
# Pytree is a container like structure like leaf pytrees, like a list of dicts or a tuple 
example_trees = [
    [1, 'a', object()],
    (1, (2, 3), ()),
    [1, {'k1': 2, 'k2': (3, 4)}, 5],
    {'a': 2, 'b': (2, 3)},
    jnp.array([1, 2, 3]),
]

# how many leaves the pytrees have.
for pytree in example_trees:
  # This `jax.tree.leaves()` method extracts the flattened leaves from the pytrees.
  leaves = jax.tree.leaves(pytree)
  logger.info(f"{repr(pytree):<45} has {len(leaves)} leaves: {leaves}") # If it's a dict then we only retrieve the values which stands as the leaves 

[INFO] [1, 'a', <object object at 0x14bf841707b0>]   has 3 leaves: [1, 'a', <object object at 0x14bf841707b0>]

[INFO] (1, (2, 3), ())                               has 3 leaves: [1, 2, 3]

[INFO] [1, {'k1': 2, 'k2': (3, 4)}, 5]               has 5 leaves: [1, 2, 3, 4, 5]

[INFO] {'a': 2, 'b': (2, 3)}                         has 3 leaves: [2, 2, 3]

[INFO] Array([1, 2, 3], dtype=int64)                 has 1 leaves: [Array([1, 2, 3], dtype=int64)]

In [37]:
list_of_lists = [
    [1, 2, 3],
    [1, 2],
    [1, 2, 3, 4]
]

jax.tree.map(lambda x: x*2, list_of_lists) # Operates on the entier tree

[[2, 4, 6], [2, 4], [2, 4, 6, 8]]

## Using JAX

In this section, we explore how far we can go using JAX alone by transforming our Python classes into structures that are compatible with JAX's functional programming model. The goal is to preserve as much of the original object-oriented design as possible while still benefiting from JAX transformations such as JIT compilation, automatic differentiation, and vectorization.

This serves as an intermediate step before introducing Equinox. By first understanding the limitations and workarounds required when using plain JAX with Python classes, we can better appreciate how Equinox simplifies the process by providing a more natural way to define and transform object-oriented models while remaining fully compatible with JAX's ecosystem.

In [38]:
from jax.tree_util import register_pytree_node_class
from scipy.io import FortranFile

@register_pytree_node_class
class Global_module_hydrol_alma:

    def __init__(self):
        self.nice = jnp.int32(8)
        self.ncirc = jnp.int32(3)
        self.nsnow = jnp.int32(3)
        self.nslm = jnp.int32(11)
        self.nvm = jnp.int32(15)
        self.nstm = jnp.int32(3)
        self.kjpindex = jnp.int32(4717)
        self.zero = jnp.float64(0.0)
        self.nnobio = jnp.int32(1)
        self.delintercept = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.delsoilmoist = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.delswe = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.humtot = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.mx_eau_var = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.snow_beg = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.snow_end = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watsoil_beg = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watsoil_end = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watveg_beg = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watveg_end = jnp.zeros((self.kjpindex,), dtype=jnp.float64)

    def declaration_initialization(self):
        print("--- add the declaration and initialization in module global ---")

        path = "/home/kardaneh/Fgpt/benchmark/hydrol_alma/global.bin"
        ffile = FortranFile(path, "r")

        fields = [
            'delintercept','delsoilmoist','delswe','humtot','mx_eau_var',
            'snow_beg','snow_end','tot_watsoil_beg','tot_watsoil_end',
            'tot_watveg_beg','tot_watveg_end',
        ]

        updated = {}
        for name in fields:
            arr = getattr(self, name)
            data = ffile.read_reals(np.float64).reshape(arr.shape, order="F")
            updated[name] = jnp.asarray(data)

        # Create fresh instance with default sizes
        new_instance = Global_module_hydrol_alma()

        # Replace only the array attributes
        for name in fields:
            setattr(new_instance, name, updated[name])

        return new_instance

    def tree_flatten(self):
        children = (self.delintercept,self.delsoilmoist, 
                    self.delswe,self.humtot,self.mx_eau_var,self.snow_beg,self.snow_end,self.tot_watsoil_beg,self.tot_watsoil_end,self.tot_watveg_beg,self.tot_watveg_end)
        aux_data = None
        return (children, aux_data)
    
    @classmethod
    def tree_unflatten(cls, aux_data, children):
        return cls(*children)
        

In [39]:
gm = Global_module_hydrol_alma()

In [40]:
gm = gm.declaration_initialization()

--- add the declaration and initialization in module global ---


In [41]:
gm.tot_watsoil_beg # got updated 

Array([599.98690234, 599.98767302, 599.98634545, ..., 597.92368896,
       599.78779044, 605.46231565], dtype=float64)

In [42]:
jax.clear_caches()

This class shows how a Python object can be made compatible with JAX using `@register_pytree_node_class`, allowing it to work with transformations like `jit`, `grad`, and `vmap`.

---

#### Advantages

### 1. JAX compatibility
By defining `tree_flatten` and `tree_unflatten`, the class can be used inside JAX transformations such as `jit`, `vmap`, and `grad`.

### 2. Structured state
It keeps related variables organized in a single object, which is useful for large scientific models.

### 3. Control over JAX behavior
You explicitly decide what is treated as data (arrays) and what is static metadata.

### 4. Flexible data loading
External data (e.g., from Fortran files) can be loaded and converted into JAX arrays before use.

---

#### Limitations

### 1. Boilerplate code
You must manually define how the object is flattened and rebuilt, which is repetitive and error-prone.

### 2. Conceptual mismatch with JAX
Even though the class looks mutable, JAX expects functional (immutable) behavior.

### 3. Easy to break
Forgetting to update `tree_flatten` when adding new fields can lead to silent bugs.

### 4. Not JAX-traceable initialization
File I/O and setup logic cannot run inside `jit` or `grad`.

### 5. Hard to scale
As models grow, maintaining PyTree logic becomes increasingly complex.

---

#### Summary

This approach works, but it becomes difficult to maintain for large projects. This is why higher-level tools like Equinox are often preferred.

## Using equinox module

Equinox provides an effective way to represent our model’s global state and update routines in JAX without requiring extensive refactoring or abandoning the original Python structure. By allowing us to cleanly separate static configuration fields from dynamic JAX-traced arrays, Equinox avoids unnecessary recompilation and keeps the model lightweight while remaining fully compatible with JIT compilation, vectorization, and automatic differentiation. Its support for class-based modules and filter_jit enables us to write natural object-oriented update methods that JAX can still optimize efficiently, and tree_at offers a functional, JAX-friendly mechanism for updating large state structures in place. This makes Equinox especially suitable for scientific models and emulator development, where we need both high-performance numerical computation and the flexibility to mix physical parameterizations with machine-learning components, all while preserving a clear, maintainable model architecture.

In [43]:
import equinox as eqx

In [44]:
eqx.clear_caches()

In [45]:
from scipy.io import FortranFile

class GlobalModuleHydrolAlma_eqx(eqx.Module):
    # --- static metadata which are not traced by JAX
    nice: int = eqx.field(static=True)
    ncirc: int = eqx.field(static=True)
    nsnow: int = eqx.field(static=True)
    nslm: int = eqx.field(static=True)
    nvm: int = eqx.field(static=True)
    nstm: int = eqx.field(static=True)
    kjpindex: int = eqx.field(static=True)
    nnobio: int = eqx.field(static=True)

    # dynamic state variables (JAX arrays)
    delintercept: jnp.ndarray
    delsoilmoist: jnp.ndarray
    delswe: jnp.ndarray
    humtot: jnp.ndarray
    mx_eau_var: jnp.ndarray
    snow_beg: jnp.ndarray
    snow_end: jnp.ndarray
    tot_watsoil_beg: jnp.ndarray
    tot_watsoil_end: jnp.ndarray
    tot_watveg_beg: jnp.ndarray
    tot_watveg_end: jnp.ndarray
    zero: float

    def __init__(self):
        self.nice = 8
        self.ncirc = 3
        self.nsnow = 3
        self.nslm = 11
        self.nvm = 15
        self.nstm = 3
        self.kjpindex = 4717
        self.nnobio = 1
        self.zero = 0.0
        self.delintercept = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.delsoilmoist = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.delswe = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.humtot = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.mx_eau_var = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.snow_beg = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.snow_end = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watsoil_beg = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watsoil_end = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watveg_beg = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watveg_end = jnp.zeros((self.kjpindex,), dtype=jnp.float64)

    
    def declaration_initialization(self):
        print(f'--- add the declaration and initialization in module global ---')
        path = f'/home/kardaneh/Fgpt/benchmark/hydrol_alma/global.bin'
        ffile = FortranFile(path, 'r')
        updates = {}
        for attr in [
            "delintercept", "delsoilmoist", "delswe", "humtot", "mx_eau_var",
            "snow_beg", "snow_end", "tot_watsoil_beg", "tot_watsoil_end",
            "tot_watveg_beg", "tot_watveg_end"
        ]:
            arr = getattr(self, attr)
            shape = arr.shape

            if arr.dtype == jnp.float64:
                raw = ffile.read_reals(np.float64)
            else:
                raw = ffile.read_ints(np.int32)

            # reshape Fortran -> JAX
            np_arr = raw.reshape(shape, order="F")

            # convert NumPy → JAX
            jax_arr = jnp.array(np_arr)

            updates[attr] = jax_arr

        return eqx.tree_at(lambda m: tuple(getattr(m, k) for k in updates.keys()),
                        self,
                        tuple(updates.values()))

    # @jax.jit <----- one of the primary problems with jit is that it requires a manual indication of which should be treated dynamically for the tracing 
    @eqx.filter_jit # <---------- THis handles the fact that all arrays(jax and numpy) are dynamic and since we used the field function to define static variables those won't be traced either 
    def hydrol_alma(self, qsintveg, snow, snow_nobio):
        
        
        tot_watveg_beg = jnp.sum(qsintveg[:self.kjpindex], axis=1)
        tot_watsoil_beg = self.humtot[:self.kjpindex]
        snow_beg = snow[:self.kjpindex] + jnp.sum(snow_nobio[:self.kjpindex], axis=1)

        return eqx.tree_at(
            lambda m: (m.tot_watveg_beg, m.tot_watsoil_beg, m.snow_beg),
            self,
            (tot_watveg_beg, tot_watsoil_beg, snow_beg)
        ) # -> This returns the updated state like a new module(pytree) that is update with the new values as such that it's an updated Pytree with updated leaves. It doesn't do a deep copy of tree itself.  

In [46]:
gmha = GlobalModuleHydrolAlma_eqx()

In [47]:
gmha = gmha.declaration_initialization()

--- add the declaration and initialization in module global ---


In [48]:
gmha.tot_watsoil_beg

Array([599.98690234, 599.98767302, 599.98634545, ..., 597.92368896,
       599.78779044, 605.46231565], dtype=float64)

In [49]:
logger.info(f"Type: {type(gmha.tot_watsoil_beg)}")

[INFO] Type: <class 'jaxlib._jax.ArrayImpl'>

In [50]:
qsintveg = jnp.zeros((gmha.kjpindex, gmha.nvm), dtype=jnp.float64)
snow = jnp.zeros((gmha.kjpindex,), dtype=jnp.float64)
snow_nobio = jnp.zeros((gmha.kjpindex, gmha.nnobio), dtype=jnp.float64)

In [51]:
primals,tangents = jax.jvp(gmha.hydrol_alma,(qsintveg,snow,snow_nobio),(jnp.ones_like(qsintveg),jnp.ones_like(snow),jnp.ones_like(snow_nobio))) # jax.jvp computes the primal output (the updated module returned by hydrol_alma)
# and the tangent output (the linearized change to the module for the given input tangents).
# primals   = the updated GlobalModuleHydrolAlma_eqx state produced by hydrol_alma
# tangents  = the corresponding tangent module, containing JVPs of each updated field
updated_state = primals          # This is the new module
updated_tangent_state = tangents # Tangent module

In [52]:
d_tot_watveg_beg = updated_tangent_state.tot_watveg_beg
d_tot_watsoil_beg = updated_tangent_state.tot_watsoil_beg
d_snow_beg = updated_tangent_state.snow_beg

In [53]:
def run_jvp():
    primals, tangents = jax.jvp(
        gmha.hydrol_alma,
        (qsintveg, snow, snow_nobio),
        (jnp.ones_like(qsintveg), jnp.ones_like(snow), jnp.ones_like(snow_nobio))
    )
    return primals, tangents

# Warmup: run once to ensure JAX has compiled everything
run_jvp()
%timeit run_jvp()

3.51 ms ± 10.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [54]:
# THis is just to check if there is any form of leak 
# with jax.checking_leaks():
    # out = gmha.hydrol_alma(qsintveg, snow, snow_nobio)

In [55]:
# The hardest part of this process is to modify the core computation funciton itself since as said before loop and conditionals are not natively supported within jax architecture like that of Python. 

# We will start with the piece of code and start adding further line as we transform the code. Let's take for loops these are jit and grad comptaible only in the case where the argument value is indenpendant 
# of the loop condition, but if is not jit compatible only grad comptaible.(fwd,rwd)

# Thus we will just manually "translate"/"transform" from python to jax compliant code 
# All the arrays that are coming from the class itself, we will remove the self. since these values once finished will 
# be reaffected directly as seen above, the variables especially those of the arrays that we are modifying are those of 
# that will be affected(lhs) assignement only. 

In [56]:
import logging 
# Here we are trying to see if we can replace for loops and conditionals into jax compatible methods as mentioned above 
class GlobalModuleHydrolAlma_eqx(eqx.Module):
    # --- static metadata which are not traced by JAX
    nice: int = eqx.field(static=True)
    ncirc: int = eqx.field(static=True)
    nsnow: int = eqx.field(static=True)
    nslm: int = eqx.field(static=True)
    nvm: int = eqx.field(static=True)
    nstm: int = eqx.field(static=True)
    kjpindex: int = eqx.field(static=True)
    nnobio: int = eqx.field(static=True)

    # dynamic state variables (JAX arrays)
    delintercept: jnp.ndarray
    delsoilmoist: jnp.ndarray
    delswe: jnp.ndarray
    humtot: jnp.ndarray
    mx_eau_var: jnp.ndarray
    snow_beg: jnp.ndarray
    snow_end: jnp.ndarray
    tot_watsoil_beg: jnp.ndarray
    tot_watsoil_end: jnp.ndarray
    tot_watveg_beg: jnp.ndarray
    tot_watveg_end: jnp.ndarray
    zero: float

    def __init__(self):
        self.nice = 8
        self.ncirc = 3
        self.nsnow = 3
        self.nslm = 11
        self.nvm = 15
        self.nstm = 3
        self.kjpindex = 4717
        self.nnobio = 1
        self.zero = 0.0
        self.delintercept = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.delsoilmoist = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.delswe = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.humtot = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.mx_eau_var = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.snow_beg = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.snow_end = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watsoil_beg = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watsoil_end = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watveg_beg = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.tot_watveg_end = jnp.zeros((self.kjpindex,), dtype=jnp.float64)

    
    def declaration_initialization(self):
        print(f'--- add the declaration and initialization in module global ---')
        path = f'/home/kardaneh/Fgpt/benchmark/hydrol_alma/global.bin'
        ffile = FortranFile(path, 'r')
        updates = {}
        for attr in [
            "delintercept", "delsoilmoist", "delswe", "humtot", "mx_eau_var",
            "snow_beg", "snow_end", "tot_watsoil_beg", "tot_watsoil_end",
            "tot_watveg_beg", "tot_watveg_end"
        ]:
            arr = getattr(self, attr)
            shape = arr.shape

            if arr.dtype == jnp.float64:
                raw = ffile.read_reals(np.float64)
            else:
                raw = ffile.read_ints(np.int32)

            # reshape Fortran -> JAX
            np_arr = raw.reshape(shape, order="F")

            # convert NumPy → JAX
            jax_arr = jnp.array(np_arr)

            updates[attr] = jax_arr

        return eqx.tree_at(lambda m: tuple(getattr(m, k) for k in updates.keys()),
                        self,
                        tuple(updates.values()))

    # @jax.jit <----- one of the primary problems with jit is that it requires a manual indication of which should be treated dynamically for the tracing 
    @eqx.filter_jit # <---------- THis handles the fact that all arrays(jax and numpy) are dynamic and since we used the field function to define static variables those won't be traced either 
    def hydrol_alma(self, lstep_init, qsintveg, snow, snow_nobio):

        def loop_fn_1():
            tot_watveg_beg = jnp.sum(qsintveg[:self.kjpindex], axis=1)
            tot_watsoil_beg = self.humtot[:self.kjpindex]
            snow_beg = snow[:self.kjpindex] + jnp.sum(snow_nobio[:self.kjpindex], axis=1)

            return tot_watveg_beg, tot_watsoil_beg, snow_beg
        
        def no_op():
            return self.tot_watveg_beg, self.tot_watsoil_beg, self.snow_beg
        
        tot_watveg_beg, tot_watsoil_beg, snow_beg = lax.cond(
            lstep_init,
            lambda : loop_fn_1(), 
            no_op  # if lstep_init is False, do nothing(keep previous values) since there's no else
        )
        
        tot_watveg_end = jnp.sum(qsintveg[:self.kjpindex], axis=1)
        tot_watsoil_end = self.humtot[:self.kjpindex]
        snow_end = snow[:self.kjpindex] + jnp.sum(snow_nobio[:self.kjpindex])
        delintercept = tot_watveg_end[:self.kjpindex] - tot_watveg_beg[:self.kjpindex]
        delsoilmoist = tot_watsoil_end[:self.kjpindex] - tot_watsoil_beg[:self.kjpindex]
        delswe = snow_end[:self.kjpindex] - snow_beg[:self.kjpindex]

        tot_watveg_beg = tot_watveg_end
        tot_watsoil_beg = tot_watsoil_end
        snow_beg = snow_end

        return eqx.tree_at(
            lambda m: (m.tot_watveg_beg, m.tot_watsoil_beg, m.snow_beg, m.delintercept,m.delsoilmoist,m.delswe,m.snow_end,m.tot_watsoil_end,m.tot_watveg_end),
            self,
            (tot_watveg_beg, tot_watsoil_beg, snow_beg,delintercept,delsoilmoist,delswe,snow_end,tot_watsoil_end,tot_watveg_end)
        )

In [57]:
gmha = GlobalModuleHydrolAlma_eqx()
gmha = gmha.declaration_initialization()

--- add the declaration and initialization in module global ---


In [ ]:
# Here all dreviations are run in respect to the inputs 
def run_jvp():
    primals, tangents = jax.jvp(
        gmha.hydrol_alma,
        (True, qsintveg, snow, snow_nobio),
        (jnp.ones_like(1,dtype=jax.float0), jnp.ones_like(qsintveg), jnp.ones_like(snow), jnp.ones_like(snow_nobio))
    )
    return primals, tangents

# run_jvp()
%timeit -n 1000 run_jvp()

In [ ]:
jnp.logical_or(jnp.sum(jnp.ones((2,))), 0) # THis is for the case when we have > and < we need to use jnp.logical_and, perhaps this is useful in the case of having multiple conditions
# , bitwise operator &, | works on jit 
logger.info(f"Result: {jnp.greater_equal(jnp.sum(jnp.ones((2,))), 0)}")

In [ ]:
import jax
import jax.numpy as jnp

# Define the matrix A and vector x
A = jnp.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
x = jnp.array([1, 1, 1])

# Define the step function for matrix-vector multiplication
def matvec_step(carry, row):
    # Each step of the scan multiplies one row of A by the vector x
    result = jnp.dot(row, x)  # dot product of row with vector
    return carry, result   # Update the carry (partial result) and return the output

# Initial carry is 0 (starting with no partial sum)
init = 0

# Apply jax.lax.scan to accumulate the dot products
result, final_state = jax.lax.scan(matvec_step, init, A)

# the result
logger.info(f"Matrix-Vector Multiplication Result: {result}")
logger.info(f"Final state : {final_state}")

In [ ]:
import jax
import jax.numpy as jnp

# Define the matrices A (m x n) and B (n x p)
A = jnp.array([[1, 2], [3, 4], [5, 6]])  # 3x2 matrix
B = jnp.array([[1, 4], [2, 5]])          # 2x2 matrix

# Define the step function for matrix-matrix multiplication
def matmul_step(carry, b_col):
    # Each step multiplies matrix A by one column of matrix B
    result = jnp.dot(A, b_col)  # dot product of A with column of B
    return carry , result  # Update the carry (partial result) and return the output

# Initial carry is a zero vector (for accumulating the results)
init = jnp.zeros(A.shape[0])

# Apply jax.lax.scan to accumulate the dot products for each column of B
result, final_state = jax.lax.scan(matmul_step, init, B.T)  # B.T to iterate over columns of B

#  the result
logger.info(f"Matrix-Matrix Multiplication Result:\n{result}")
logger.info(f'FInal state of the output :\n{final_state}')


## Automatic differentiation

In the context of using Equinox and JAX, the goal is to determine whether model parameters are differentiable with respect to both the inputs and the model itself during backpropagation. The key difference between JVP and VJP lies in how these derivatives are computed and whether model parameters are treated as constant or differentiable.

**JVP (Jacobian-Vector Product):**
JVP is used when you want to compute the derivative of a function with respect to a specific vector (the tangent vector). If a tangent vector is provided, model parameters are differentiated with respect to it, meaning the model parameters contribute to the differentiation.  
If no specific tangent is provided, the model parameters are treated as constants during the differentiation process. Essentially, the model parameters are not part of the computation for the Jacobian-vector product in this case.

**VJP (Vector-Jacobian Product):**
VJP, on the other hand, computes the derivative of the output with respect to both the inputs and model parameters.  The VJP mechanism is particularly useful because it explicitly returns the gradients with respect to both the inputs and model parameters, making it easier to compute and handle.  
Since VJP considers model parameters as part of the differentiation process, it simplifies the flow by calculating the derivative for both the parameters and the inputs simultaneously.

**In Summary:**
- JVP is more flexible but requires a tangent vector to decide whether model parameters are included in the differentiation.

- VJP is simpler because it computes gradients for both inputs and parameters directly, making it easier to work with in practice, especially in frameworks like JAX and Equinox.

In [ ]:
class Test(eqx.Module):

    # Model parameters 
    X : jnp.ndarray
    Y : jnp.ndarray

    def __init__(self):
        # THis will be keep the self.X and self.Y as non static which means that it will be part of the Pytrees and 
        # will differentiate with them meaning alll the transformation: jit and grad are applied : https://docs.kidger.site/equinox/api/module/advanced_fields/#equinox.field
        # BY DEFAULT ALL VALUES ARE SET AS NON STATIC static = False unless specified 
        self.X = jnp.ones((3,2),dtype=jnp.float64)
        self.Y = jnp.ones((3,2),dtype=jnp.float64)
        
    @eqx.filter_jit
    def test(self, x: jnp.ndarray, y: jnp.ndarray):
        # Defining polynomial terms (could be learned parameters or constants)
        a = 1.0
        b = 1.0
        c = 1.0
        d = 1.0
        e = 1.0
        f = 0.0
        
        # Polynomial expression using x, y, X, and Y
        z = a * x**2 + b * y**2 + c * self.X * self.Y + d * x * self.X + e * y * self.Y + f
        return z

In [ ]:
# Instantiate the class
test_instance = Test()
leaves, treedef = jax.tree_util.tree_flatten(test_instance)
assert jnp.array_equal(leaves[0], test_instance.X)
assert jnp.array_equal(leaves[1], test_instance.Y) # leaves contains the differentiable PyTree fields of the module.
# If "X" appears here, then X is a differentiable parameter (NOT static).

In [ ]:
# To differentiate wrt to the inputs x and y, here self.X and self.Y will be treated as constants since we didn't provide tangents for them. 
x = jnp.ones((3,2))
y = jnp.ones((3,2))
def f(x, y):
    return test_instance.test(x, y)

primals = (x, y)
tangents = (jnp.ones((3,2),dtype=jnp.float64), 2*jnp.ones((3,2),dtype=jnp.float64))   # define dx and dy

y_primal, y_tangent = jax.jvp(f, primals, tangents) # Primal here is just the output of the funtion 
logger.info(f"Test function output: {y_primal}")
logger.info(f"Tangent linear with respect to the inputs x, y: : {y_tangent}")

In [ ]:
# To differentiate wrt X and Y, we need to send the tangent for all the arguments 
def f(model, x, y):
    return model.test(x, y)
primals = (test_instance, x, y)
tangent_model = eqx.tree_at(lambda m: (m.X, m.Y),
                            test_instance,
                            (jnp.ones((3,2),dtype=jnp.float64), 2*jnp.ones((3,2),dtype=jnp.float64)) )
tangents = (tangent_model, jnp.zeros((3,2),dtype=jnp.float64), jnp.zeros((3,2),dtype=jnp.float64))

y_primal, y_tangent = jax.jvp(f, primals, tangents)
logger.info(f"Test function output:\n {y_primal}")
logger.info(f"Tangent linear with respect to the inputs x, y and the model parameters X,Y :\n {y_tangent}")

In [ ]:
# VJP on the other hand is much less complicated thatn this the gradients are done with respect to all differntial arguments inputs and model parameters 
def f(model, x, y):
    return model.test(x, y)

y_val, vjp_fn = jax.vjp(f, test_instance, x, y)
logger.info(f'Y_val is just like the primals from above:\n {y_val}')
logger.info(f'vjp_fn is the the cotagent vector representing the vector jacobian product:\n {vjp_fn}') # THis will be used to compute the VJP of the cotangent 

In [ ]:
cotangent = jnp.ones((3,2))

dmodel, dx, dy = vjp_fn(cotangent) # Here there are three ouputs which 
# dmodel the model parameters differentiable, then the differentiable with respect to the inputs x and y 
logger.info(f'dmodel, dtest/dself.X and dtest/dself.Y: {dmodel}')
logger.info(f'dtest/dself.X:\n {dmodel.X}')
logger.info(f'dtest/dself.Y:\n {dmodel.Y}')
logger.info(f'dtest/dx :\n {dx}')
logger.info(f'dtest/dy :\n {dy}')